# dARK Minter API - Test Notebook

This notebook tests the `dark-core-minter-api` endpoints using FastAPI's TestClient.

It covers:
1.  **Setup**: Initializing the Orchestrator and TestClient.
2.  **Health Check**: Verifying API availability.
3.  **Authority**: Setting up a test authority and querying it via API.
4.  **Minting**: Creating (Minting) new ARKs via the API.

**Note**: This API does NOT support resolving or looking up ARKs.


## 1. Setup Environment

In [1]:
import os
import sys
import logging
from dotenv import load_dotenv

# Add app to path
sys.path.append(os.path.abspath('..'))

# Logging
logging.basicConfig(level=logging.INFO)

# Load unified configuration
if os.path.exists('../../.env'):
    load_dotenv('../../.env')
    print("Loaded ../../.env")
elif os.path.exists('../.env'):
    load_dotenv('../.env')
    print("Loaded ../.env")
else:
    print('⚠️  Warning: No .env found')

Loaded ../../.env


## 2. Initialize App & TestClient

In [2]:
from fastapi.testclient import TestClient
from app.main import app
from app.dependencies import get_orchestrator, init_orchestrator

# Initialize the orchestrator manually before creating TestClient
# This is needed because TestClient may not always trigger lifespan events
try:
    orchestrator = get_orchestrator()
    print("✅ Orchestrator already initialized")
except RuntimeError:
    print("Initializing orchestrator manually...")
    init_orchestrator()
    print("✅ Orchestrator initialized")

# Create TestClient
client = TestClient(app)

print("✅ TestClient initialized")

/Users/lmatas/source/dark/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
INFO:app.dependencies:Initializing DARKOrchestrator...
INFO:dark_orchestrator.client:Connecting to blockchain at http://localhost:8545
INFO:dark_orchestrator.client:Connected! Block number: 2940
INFO:dark_orchestrator.client:Admin account: 0xf17f52151EbEF6C7334FAD080c5704D77216b732
INFO:dark_orchestrator.client:DARKOrchestrator initialized successfully
INFO:app.dependencies:DARKOrchestrator initialized successfully


Initializing orchestrator manually...
✅ Orchestrator initialized
✅ TestClient initialized


## 3. Health Check

In [3]:
response = client.get("/health")
print(f"Status Code: {response.status_code}")
print(f"Response: {response.json()}")
assert response.status_code == 200

INFO:httpx:HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


Status Code: 200
Response: {'status': 'healthy', 'blockchain_connected': True, 'current_block': 2941}


## 4. Setup Test Authority (Direct Orchestrator Access)

We need a registered authority to test minting. We'll use the orchestrator directly to set one up, similar to the lib test.

In [4]:
import time

# Get the orchestrator instance used by the app
orchestrator = get_orchestrator()

# Setup Authority
UUID = f"minter-test-{int(time.time())}"
NAAN = "12345"
NAANS = [NAAN]

print(f"Setting up authority: {UUID}")
authority = orchestrator.setup_authority(UUID, NAANS)
print(f"✅ Authority Setup: {authority.wallet_address}")

INFO:dark_orchestrator.client:Setting up authority: minter-test-1769014733 with NAANs: ['12345']
INFO:dark_orchestrator.client:UUID 'minter-test-1769014733' is available, proceeding with registration
INFO:dark_orchestrator.client:Created new wallet for minter-test-1769014733: 0x6524859EdC6b5b853a45797d86a3c9d035189b4E
INFO:dark_orchestrator.client:Encrypted private key for blockchain storage
INFO:dark_orchestrator.client:Funding wallet with 0.01 ETH


Setting up authority: minter-test-1769014733


INFO:dark_orchestrator.client:Funded 0x6524859EdC6b5b853a45797d86a3c9d035189b4E with 0.01 ETH
INFO:dark_orchestrator.client:Registering authority minter-test-1769014733 with wallet 0x6524859EdC6b5b853a45797d86a3c9d035189b4E
INFO:dark_orchestrator.authority:Registering authority 'minter-test-1769014733' with wallet 0x6524859EdC6b5b853a45797d86a3c9d035189b4E
INFO:dark_orchestrator.authority:TX sent: 99ce11324ceff7c15ec81496302ab8aaeadccf06460fe921501fa6581d082c40
INFO:dark_orchestrator.authority:TX confirmed. Gas used: 209038
INFO:dark_orchestrator.client:Authority minter-test-1769014733 registered successfully
INFO:dark_orchestrator.authority:Authorizing NAAN '12345' for 0x6524859EdC6b5b853a45797d86a3c9d035189b4E
INFO:dark_orchestrator.authority:TX sent: 6fec5df012d71c5f4483ca2edd5342b2c42e269c79a3cd564b632ea57ddbe2d8
INFO:dark_orchestrator.authority:TX confirmed. Gas used: 96922


✅ Authority Setup: 0x6524859EdC6b5b853a45797d86a3c9d035189b4E


## 5. Test Authority Endpoint

Verify the API can retrieve the authority we just created.

In [5]:
response = client.get(f"/api/v1/authority/{UUID}")

print(f"Status: {response.status_code}")
print(f"Body: {response.json()}")

assert response.status_code == 200
data = response.json()
assert data['uuid'] == UUID
assert NAAN in data['naans']

INFO:app.api.authority:Getting authority: minter-test-1769014733


INFO:httpx:HTTP Request: GET http://testserver/api/v1/authority/minter-test-1769014733 "HTTP/1.1 200 OK"


Status: 200
Body: {'uuid': 'minter-test-1769014733', 'naans': ['12345'], 'active': True}


## 6. Test Batch Mint Endpoint

Use the `/api/v1/mint/batch` endpoint to mint a new ARK.

In [6]:
NAME = f"test-doc-{int(time.time())}"
URL = "https://example.org/doc/1"
CID = "bafybeigdyrzt5sfp7udbbkc5dla2yv5ifyrkkwdxgper"

payload = {
    "items": [
        {
            "authority_id": UUID,
            "naan": NAAN,
            "name": NAME,
            "url": URL,
            "cid": CID
        }
    ]
}

print(f"Minting ark:/{NAAN}/{NAME} ...")
response = client.post("/api/v1/mint/batch", json=payload)

print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

assert response.status_code == 200
result = response.json()
assert result['status'] == 'ok'
assert result['results'][0]['status'] == 'success'

INFO:app.api.mint:Processing batch mint with 1 items
INFO:dark_orchestrator.ark:Creating ARK: ark:/12345/test-doc-1769014790
INFO:dark_orchestrator.ark:TX sent: 880367600755b5b0608311ac2098b362ee6c3b2773d69a3ce9b68eb25eb54207


Minting ark:/12345/test-doc-1769014790 ...


INFO:dark_orchestrator.ark:TX confirmed. Gas used: 246271
INFO:app.api.mint:Created ARK: ark:/12345/test-doc-1769014790
INFO:app.api.mint:Batch mint complete: 1 success, 0 errors
INFO:httpx:HTTP Request: POST http://testserver/api/v1/mint/batch "HTTP/1.1 200 OK"


Status: 200
Response: {'status': 'ok', 'results': [{'dark_id': 'ark:/12345/test-doc-1769014790', 'name': 'test-doc-1769014790', 'status': 'success', 'transaction_ref': None, 'error': None}]}
